# Actividad 3.1 (Valores Atípicos)
Archivo: Citas_Digital_Filtrado.csv

In [ ]:
#Importamos las librerias pandas, numpy y matplotlib respectivamente
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Carga desde un archivo .csv sin indice
data = pd.read_csv('Citas_Digital_Filtrado.csv')
data.info()

In [ ]:
#Corroboramos valores nulos
valores_nulos=data.isnull().sum()
valores_nulos

**PROCEDIMIENTO PARA ELIMINAR VALORES NULOS EN DATAFRAME**

In [ ]:
#Eliminar columnas innecesarias (practicamente vacias)
#axis = 1 para columnas
data0 = data.drop(['Telefono ','Unnamed: 15'], axis=1)

In [ ]:
#Reemplazamos valores nulos del dataframe con "bfill"
data1 = data0.bfill()
data1

**Las columnas "¿Por que...?" quedan con nulos porque bfill no tiene nada abajo de donde copiar en las ultimas filas. Igual que en la Actividad 2.1, se sustituyen con un valor string fijo ("No aplica")**

In [ ]:
#Sustituir valores nulos por un string en concreto (igual que en Actividad 2.1)
data1['¿Por que no ha visitado la agencia?'] = data1['¿Por que no ha visitado la agencia?'].fillna('No aplica')
data1['¿Por que no hay solicitud de credito?'] = data1['¿Por que no hay solicitud de credito?'].fillna('No aplica')
data1['¿Por que no hubo prueba de manejo ?'] = data1['¿Por que no hubo prueba de manejo ?'].fillna('No aplica')
data1['¿Por que no hubo proceso wow?'] = data1['¿Por que no hubo proceso wow?'].fillna('No aplica')
data1['¿Por que se asigno antes de vistar piso?'] = data1['¿Por que se asigno antes de vistar piso?'].fillna('No aplica')

In [ ]:
#Corroboramos valores nulos del dataframe
valores_nulos=data1.isnull().sum()
valores_nulos

In [ ]:
#Creo 2 dataframe para poder procesar los outliers
cuantitativas= data1[['PDM','SDC','Venta']]
cualitativas= data1[['BDC ','Fecha','Nombre Cliente ','Estatus de Lead','Potencial de compra',
                      'Asesor Asignado','¿Por que no ha visitado la agencia?',
                      '¿Por que no hay solicitud de credito?','¿Por que no hubo prueba de manejo ?',
                      '¿Por que no hubo proceso wow?','¿Por que se asigno antes de vistar piso?']]

In [ ]:
#Realizamos diagrama de caja o bigote de cada columna del dataframe
fig = plt.figure(figsize =(15, 8))
cuantitativas.plot(kind='box', vert=False)
plt.title("Valores Atípicos - datos originales")
plt.show() #dibujamos el diagrama

**PROCEDIMIENTO "DESVIACIÓN ESTÁNDAR" PARA ELIMINAR OUTLIERS EN DATAFRAME**

In [ ]:
#Método aplicando desviación estandar. Encuentro los valores extremos
y=cuantitativas
Limite_Superior= y.mean() + 3*y.std()
Limite_Inferior= y.mean() - 3*y.std()
print("Limite superior permitido", Limite_Superior)
print("Limite inferior permitido", Limite_Inferior)

In [ ]:
#Obtenemos datos y los outliers se convierten en nulos en el DataFrame
data3= cuantitativas[(y<=Limite_Superior)&(y>=Limite_Inferior)]
data3

In [ ]:
#Corroboramos valores nulos del dataframe
valores_nulos=data3.isnull().sum()
valores_nulos

In [ ]:
#Reemplazamos valores atípicos (nulos) del dataframe con "mean"
#Realizamos una copia del dataframe
data_clean=data3.copy()
data_clean=data_clean.fillna(round(data3.mean(),1))
data_clean

In [ ]:
#Corroboramos valores nulos del dataframe LIMPIO
valores_nulos=data_clean.isnull().sum()
valores_nulos

In [ ]:
#Diagrama de caja despues de tratar outliers (desviacion estandar)
fig = plt.figure(figsize =(15, 8))
data_clean.plot(kind='box', vert=False)
plt.title("Valores Atípicos - despues de Desviacion Estandar")
plt.show()

**Convertir DataSet (Desviación Estándar) a CSV**

In [ ]:
# Unimos el dataframe cuantitativo limpio con el dataframe cualitativo
Datos_limpios_std = pd.concat([cualitativas, data_clean], axis=1)
Datos_limpios_std

In [ ]:
#Corroboramos valores nulos del dataframe LIMPIO
valores_nulos=Datos_limpios_std.isnull().sum()
valores_nulos

In [ ]:
#Convertir DataFrame a CSV
Datos_limpios_std.to_csv("Citas_Digital_Sin_Outliers_DesviacionEstandar.csv", index=False)

**PROCEDIMIENTO "CUANTILES" PARA SUSTITUIR OUTLIERS EN DATAFRAME**

In [ ]:
#Método aplicando Cuartiles. Encuentro cuartiles 0.25 y 0.75
y=cuantitativas

percentile25=y.quantile(0.25) #Q1
percentile75=y.quantile(0.75) #Q3
iqr= percentile75 - percentile25

Limite_Superior_iqr= percentile75 + 1.5*iqr
Limite_Inferior_iqr= percentile25 - 1.5*iqr
print("Limite superior permitido", Limite_Superior_iqr)
print("Limite inferior permitido", Limite_Inferior_iqr)

In [ ]:
#Obtenemos datos limpios del Dataframe
data3_iqr= cuantitativas[(y<=Limite_Superior_iqr)&(y>=Limite_Inferior_iqr)]
data3_iqr

In [ ]:
#Corroboramos valores nulos del dataframe LIMPIO
valores_nulos=data3_iqr.isnull().sum()
valores_nulos

In [ ]:
#Reemplazamos valores atípicos (nulos) del dataframe con "median"
#Realizamos una copia del dataframe
data4_iqr=data3_iqr.copy()
data4_iqr=data4_iqr.fillna(round(data3_iqr.median(),1))
data4_iqr

In [ ]:
#Corroboramos valores nulos del dataframe LIMPIO
valores_nulos=data4_iqr.isnull().sum()
valores_nulos

In [ ]:
#Diagrama de caja despues de tratar outliers (IQR)
fig = plt.figure(figsize =(15, 8))
data4_iqr.plot(kind='box', vert=False)
plt.title("Valores Atípicos - despues de IQR")
plt.show()

**Convertir DataSet (Rango Intercuartílico) a CSV**

In [ ]:
# Unimos el dataframe cuantitativo limpio con el dataframe cualitativo
Datos_limpios_iqr = pd.concat([cualitativas, data4_iqr], axis=1)
Datos_limpios_iqr

In [ ]:
#Corroboramos valores nulos del dataframe LIMPIO
valores_nulos=Datos_limpios_iqr.isnull().sum()
valores_nulos

In [ ]:
#Convertir DataFrame a CSV
Datos_limpios_iqr.to_csv("Citas_Digital_Sin_Outliers_IQR.csv", index=False)